# PixArt-alpha -- Fine-tuning DiT (Diffusion Transformer) + LoRA


## 0. Verificar GPU

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU no encontrada — usando CPU.")
    
print(f"\nDevice: {device}")

GPU : Tesla T4
VRAM: 15.6 GB

Device: cuda


## 1. Instalar dependencias

In [ ]:
!pip install -q diffusers transformers peft datasets accelerate matplotlib torchvision sentencepiece protobuf tiktoken
!pip install -q bitsandbytes>=0.41.0
!pip install -q -U torchao
print("Dependencias listas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.3 MB/s eta 0:00:00
Dependencias listas


## 2. Directorios de salida

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR  = "/content/drive/MyDrive/pixart_lora"   # modelo → Drive (persiste)
CACHE_DIR = "/content/emb_cache"                   # embeddings → disco local (más rápido)
os.makedirs(SAVE_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Modelo -> {SAVE_DIR}")
print(f"Cache  -> {CACHE_DIR}")

Mounted at /content/drive
Modelo -> /content/drive/MyDrive/pixart_lora
Cache  -> /content/emb_cache


## 3. Imports y configuracion

In [ ]:
import random, gc, json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from tqdm.notebook import tqdm

MODEL_ID    = "PixArt-alpha/PixArt-XL-2-512x512"
EPOCHS      = 1
LR          = 1e-4
BATCH_SIZE  = 2
GRAD_ACCUM  = 8
MAX_SAMPLES = 5000
IMAGE_SIZE  = 512
MAX_SEQ_LEN = 64    # Flickr8k captions ~10-20 palabras; 64 tokens cubre el 99%
LORA_RANK   = 8
LORA_ALPHA  = 32
print(f"Modelo: {MODEL_ID} | Batch efectivo: {BATCH_SIZE*GRAD_ACCUM} | r={LORA_RANK}")

Modelo: PixArt-alpha/PixArt-XL-2-512x512 | Batch efectivo: 16 | r=8


## 4. Fase 1 -- Pre-computar embeddings T5-XXL



In [ ]:
from transformers import T5EncoderModel, AutoTokenizer, BitsAndBytesConfig

# 8-bit: T5-XXL (~11 GB int8) cabe entero en la T4 (15.6 GB VRAM) → 0 RAM de CPU
bnb_8bit = BitsAndBytesConfig(load_in_8bit=True)
print("Cargando T5-XXL en 8-bit (~11 GB VRAM, sin tocar RAM)...")
tokenizer_t5 = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
t5_encoder   = T5EncoderModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder",
    quantization_config=bnb_8bit,
    device_map="auto", low_cpu_mem_usage=True,
)
t5_encoder.eval()
print(f"T5 cargado. VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB")

print("Cargando Flickr8k...")
raw_ds   = load_dataset("jxie/flickr8k", split="train")
raw_ds   = raw_ds.select(range(min(MAX_SAMPLES, len(raw_ds))))
cap_cols = [c for c in raw_ds[0].keys() if c.startswith("caption")]
captions = [random.choice([raw_ds[i][c] for c in cap_cols]) for i in range(len(raw_ds))]
print(f"Muestras: {len(raw_ds)}")

# Streaming a disco con numpy memmap → 0 acumulación en RAM
n        = len(captions)
emb_gb   = n * MAX_SEQ_LEN * 4096 * 2 / 1e9
emb_path = f"{CACHE_DIR}/t5_embs.npy"
msk_path = f"{CACHE_DIR}/t5_masks.npy"
emb_mmap = np.memmap(emb_path, dtype='float16', mode='w+', shape=(n, MAX_SEQ_LEN, 4096))
msk_mmap = np.memmap(msk_path, dtype='int32',   mode='w+', shape=(n, MAX_SEQ_LEN))
print(f"Escribiendo {n} embeddings a disco ({emb_gb:.2f} GB float16)...")

ENCODE_BATCH = 4
with torch.no_grad():
    for i in tqdm(range(0, n, ENCODE_BATCH)):
        end  = min(i + ENCODE_BATCH, n)
        toks = tokenizer_t5(
            captions[i:end], padding="max_length", max_length=MAX_SEQ_LEN,
            truncation=True, return_tensors="pt"
        )
        out = t5_encoder(
            input_ids=toks.input_ids.to(t5_encoder.device),
            attention_mask=toks.attention_mask.to(t5_encoder.device)
        )
        emb_mmap[i:end] = out.last_hidden_state.cpu().half().numpy()
        msk_mmap[i:end] = toks.attention_mask.numpy().astype('int32')
        del out, toks

emb_mmap.flush(); msk_mmap.flush()
del emb_mmap, msk_mmap
print(f"Embeddings en disco: {emb_gb:.2f} GB")

del t5_encoder; gc.collect(); torch.cuda.empty_cache()
vram = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"T5 descargado. VRAM libre: {vram:.1f} GB")

Cargando T5-XXL en 8-bit (~11 GB VRAM, sin tocar RAM)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer/spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Muestras: 5000
Escribiendo 5000 embeddings a disco (2.62 GB float16)...


  0%|          | 0/1250 [00:00<?, ?it/s]

Embeddings en disco: 2.62 GB
T5 descargado. VRAM libre: 15.6 GB


## 5. Cargar DiT + VAE + aplicar LoRA



In [ ]:
from diffusers import AutoencoderKL, DDPMScheduler
from diffusers.models import Transformer2DModel
from peft import LoraConfig, get_peft_model

transformer = Transformer2DModel.from_pretrained(
    MODEL_ID, subfolder="transformer", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float16)
vae.requires_grad_(False)
vae.to(device)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.1, bias="none",
)
transformer = get_peft_model(transformer, lora_config)
transformer.print_trainable_parameters()
transformer.to(device)
print(f"VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

transformer/diffusion_pytorch_model.safe(…):   0%|          | 0.00/2.45G [00:00<?, ?B/s]

Some weights of the model checkpoint at PixArt-alpha/PixArt-XL-2-512x512 were not used when initializing PixArtTransformer2DModel: 
 ['caption_projection.y_embedding']


config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

trainable params: 4,128,768 || all params: 614,984,864 || trainable%: 0.6714
VRAM usada: 1.4 GB


## 6. Dataset

In [ ]:
class PixArtDataset(Dataset):
    def __init__(self, hf_ds, embeddings, masks, transform):
        self.ds=hf_ds; self.embeddings=embeddings
        self.masks=masks; self.transform=transform
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        return {"pixel_values": self.transform(self.ds[idx]["image"].convert("RGB")),
                "encoder_hidden_states": self.embeddings[idx],
                "encoder_attention_mask": self.masks[idx]}

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

n = len(raw_ds)
t5_embs  = torch.from_numpy(
    np.memmap(f"{CACHE_DIR}/t5_embs.npy", dtype='float16', mode='r',
              shape=(n, MAX_SEQ_LEN, 4096)).copy()
)
t5_masks = torch.from_numpy(
    np.memmap(f"{CACHE_DIR}/t5_masks.npy", dtype='int32', mode='r',
              shape=(n, MAX_SEQ_LEN)).copy()
)
print(f"Embeddings en RAM: {t5_embs.nbytes/1e9:.2f} GB (float16)")

dataset    = PixArtDataset(raw_ds, t5_embs, t5_masks, transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True)
print(f"Dataset: {len(dataset)} muestras | {len(dataloader)} batches/ep")

Embeddings en RAM: 2.62 GB (float16)
Dataset: 5000 muestras | 2500 batches/ep


## 7. Entrenamiento LoRA sobre DiT




In [ ]:
optimizer = torch.optim.AdamW(
    transformer.parameters(),
    lr=LR,
    weight_decay=1e-2
)

scaler = torch.amp.GradScaler("cuda")

transformer.train()

for epoch in range(EPOCHS):

    print(f"\n-- Epoca {epoch+1}/{EPOCHS} --")
    total_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(dataloader)):
        pv = batch["pixel_values"].to(
            device,
            dtype=torch.float16
        )

        enc_hs = batch["encoder_hidden_states"].to(
            device,
            dtype=torch.float16
        )

        enc_am = batch["encoder_attention_mask"].to(device)
        # VAE
        with torch.no_grad():

            latents = (
                vae.encode(pv)
                .latent_dist.sample()
                * vae.config.scaling_factor
            )

        # Noise
        noise = torch.randn_like(latents)

        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],),
            device=device
        ).long()

        noisy = noise_scheduler.add_noise(
            latents,
            noise,
            timesteps
        )

        # Forward
        with torch.amp.autocast("cuda"):

            pred = transformer(
                hidden_states=noisy,
                timestep=timesteps,
                encoder_hidden_states=enc_hs,
                encoder_attention_mask=enc_am,
                return_dict=False
            )[0]

            # Mantener solo canales de ruido
            pred = pred[:, :4]

            loss = F.mse_loss(
                pred.float(),
                noise.float()
            )

        # Backprop
        scaler.scale(
            loss / GRAD_ACCUM
        ).backward()

        if (step + 1) % GRAD_ACCUM == 0:

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad()

        total_loss += loss.item()

    print(
        f"Loss promedio: "
        f"{total_loss / len(dataloader):.4f}"
    )

print("Entrenamiento terminado")


-- Epoca 1/1 --


  0%|          | 0/2500 [00:00<?, ?it/s]

Loss promedio: 0.1387
Entrenamiento terminado


## 8. Guardar pesos LoRA

In [ ]:
transformer.save_pretrained(f"{SAVE_DIR}/lora_transformer")
config = {"model_id": MODEL_ID, "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
          "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE,
          "grad_accum": GRAD_ACCUM, "max_samples": MAX_SAMPLES}
with open(f"{SAVE_DIR}/config.json", "w") as f: json.dump(config, f, indent=2)
print(f"Guardado en {SAVE_DIR}")

Guardado en /content/drive/MyDrive/pixart_lora
